# Quick-Start: Text Attribution with Captum

**Goal:** Explain any transformer text classifier prediction in under 2 minutes.

**What you get:** Colored token attribution (green=supports, red=opposes) for any HuggingFace classifier.

**Prerequisites:**
```bash
pip install captum transformers
```

---

In [ ]:
import sys; sys.path.insert(0, '../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
# ============================================================
# STEP 1: Load Model
# ============================================================
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from captum.attr import LayerIntegratedGradients

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- CONFIGURE HERE: swap for any HuggingFace classifier ---
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
# Other options:
# "cardiffnlp/twitter-roberta-base-sentiment-latest"  (3-class: neg/neu/pos)
# "ProsusAI/finbert"  (financial sentiment)

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

LABEL_MAP = model.config.id2label
print(f"Ready. Labels: {LABEL_MAP}")

In [ ]:
# ============================================================
# STEP 2: Configure LayerIntegratedGradients
# ============================================================

# --- CONFIGURE HERE: target the embedding layer of your model ---
# DistilBERT:
embedding_layer = model.distilbert.embeddings
# BERT:
# embedding_layer = model.bert.embeddings
# RoBERTa:
# embedding_layer = model.roberta.embeddings

lig = LayerIntegratedGradients(model, embedding_layer)
print(f"LayerIG targeting: {embedding_layer.__class__.__name__}")

In [ ]:
# ============================================================
# STEP 3: Attribution Function
# ============================================================

def explain_text(text, target_class=None, n_steps=50):
    """
    Compute per-token attribution for a text input.

    Parameters
    ----------
    text : str
    target_class : int or None  (None = use top predicted class)
    n_steps : int

    Returns
    -------
    tokens : list of str
    attributions : ndarray (seq_len,)
    pred_label : str
    pred_prob : float
    delta : float
    """
    inputs = tokenizer(text, return_tensors='pt')
    input_ids     = inputs['input_ids'].to(DEVICE)
    attention_mask = inputs['attention_mask'].to(DEVICE)

    with torch.no_grad():
        logits = model(**inputs.to(DEVICE)).logits
        probs = torch.softmax(logits, dim=-1)
        pred_cls  = probs.argmax().item()
        pred_prob = probs.max().item()

    if target_class is None:
        target_class = pred_cls

    baseline_ids = torch.zeros_like(input_ids)

    attr, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        target=target_class,
        additional_forward_args=(attention_mask,),
        n_steps=n_steps,
        return_convergence_delta=True
    )

    # Sum across embedding dimension → per-token score
    token_attr = attr.sum(dim=-1).squeeze(0).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

    return tokens, token_attr, LABEL_MAP[pred_cls], pred_prob, delta.item()


def visualize_tokens(tokens, attrs, title='', pred_label='', pred_prob=0.0, delta=0.0):
    """Colored token attribution visualization."""
    # Remove special tokens
    disp_tokens = []
    disp_attrs  = []
    for tok, att in zip(tokens, attrs):
        if tok not in ['[CLS]', '[SEP]', '[PAD]']:
            disp_tokens.append(tok.replace('##', ''))
            disp_attrs.append(att)

    max_abs = max(abs(a) for a in disp_attrs) + 1e-8
    norm = [a / max_abs for a in disp_attrs]

    n = len(disp_tokens)
    fig_w = max(10, n * 0.95)
    fig, (ax_text, ax_bar) = plt.subplots(
        2, 1, figsize=(fig_w, 3.5),
        gridspec_kw={'height_ratios': [2, 1]}
    )

    # Text panel
    ax_text.set_xlim(0, n + 0.5)
    ax_text.set_ylim(0, 1)
    ax_text.axis('off')
    x = 0.4
    for tok, score in zip(disp_tokens, norm):
        intensity = min(abs(score) * 1.2 + 0.1, 0.95)
        fc = (0.0, intensity * 0.9, 0.0, 0.85) if score > 0 else (intensity * 0.9, 0.0, 0.0, 0.85)
        ax_text.text(x, 0.5, tok, ha='center', va='center',
                      fontsize=12, fontweight='bold', color='white',
                      bbox=dict(boxstyle='round,pad=0.3', facecolor=fc,
                                edgecolor='none', alpha=0.9))
        x += max(len(tok) * 0.2 + 0.4, 0.8)
    ax_text.set_title(
        f'{title}\nPrediction: {pred_label} ({pred_prob:.1%}) | δ={delta:.4f}',
        fontsize=11
    )

    # Bar panel
    bar_colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in norm]
    ax_bar.bar(range(n), norm, color=bar_colors, alpha=0.8)
    ax_bar.set_xticks(range(n))
    ax_bar.set_xticklabels(disp_tokens, rotation=30, ha='right', fontsize=8)
    ax_bar.axhline(0, color='black', lw=0.8)
    ax_bar.set_ylabel('Attribution')
    patches = [
        mpatches.Patch(color='#2ecc71', label='Supports prediction'),
        mpatches.Patch(color='#e74c3c', label='Opposes prediction')
    ]
    ax_bar.legend(handles=patches, fontsize=8, loc='upper right')

    plt.tight_layout()
    plt.show()

print("Functions ready.")

In [ ]:
# ============================================================
# STEP 4: Explain Your Text
# ============================================================

# --- CONFIGURE HERE: replace with your text ---
texts_to_explain = [
    "This movie was absolutely fantastic and I loved every moment.",
    "Terrible acting and a boring predictable plot. Complete waste of time.",
    "Some parts were great but the ending was disappointing.",
]

for text in texts_to_explain:
    tokens, attrs, pred_label, pred_prob, delta = explain_text(text)
    print(f"Text: '{text[:50]}...'  →  {pred_label} ({pred_prob:.1%}), δ={delta:.4f}")
    visualize_tokens(
        tokens, attrs,
        title=f'Text: "{text[:60]}"',
        pred_label=pred_label, pred_prob=pred_prob, delta=delta
    )

In [ ]:
# ============================================================
# STEP 5: Dual-Class Attribution (Optional)
# ============================================================
# For ambiguous text, compare attribution for BOTH classes

ambiguous_text = "Some parts were great but the ending was disappointing."

# n_classes from model config
n_classes = len(LABEL_MAP)

if n_classes == 2:
    tok0, att0, _, _, _ = explain_text(ambiguous_text, target_class=0)
    tok1, att1, _, _, _ = explain_text(ambiguous_text, target_class=1)

    # Filter special tokens
    display_mask = [t not in ['[CLS]', '[SEP]', '[PAD]'] for t in tok0]
    disp_tokens  = [t.replace('##', '') for t, m in zip(tok0, display_mask) if m]
    attrs0 = np.array([a for a, m in zip(att0, display_mask) if m])
    attrs1 = np.array([a for a, m in zip(att1, display_mask) if m])

    max_abs = max(np.abs(attrs0).max(), np.abs(attrs1).max()) + 1e-8
    attrs0_n = attrs0 / max_abs
    attrs1_n = attrs1 / max_abs

    n = len(disp_tokens)
    x = np.arange(n)
    w = 0.4

    inputs_viz = tokenizer(ambiguous_text, return_tensors='pt')
    with torch.no_grad():
        probs_viz = torch.softmax(model(**inputs_viz.to(DEVICE)).logits, dim=-1)[0]

    fig, ax = plt.subplots(figsize=(max(10, n * 0.85), 5))
    ax.bar(x - w/2, attrs0_n, w, label=f'{LABEL_MAP[0]}', color='#e74c3c', alpha=0.8)
    ax.bar(x + w/2, attrs1_n, w, label=f'{LABEL_MAP[1]}', color='#2ecc71', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(disp_tokens, rotation=30, ha='right', fontsize=9)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('Normalized Attribution')
    ax.set_title(
        f'Dual-Class Attribution — Ambiguous Sentence\n'
        f'"{ambiguous_text[:60]}"\n'
        f'{LABEL_MAP[0]}: {probs_viz[0]:.1%} | {LABEL_MAP[1]}: {probs_viz[1]:.1%}'
    )
    ax.legend()
    plt.tight_layout()
    plt.show()

print("\nQuick-start complete.")
print("Copy explain_text() and visualize_tokens() into your own code.")